# Map MorphSeq embryos to sequencing embryos

One fish-sized table, one readable Excel lookup, and no filtering. 🐟

The plate Excel files define the identity mapping. Prediction and sequencing-QC files only add annotations.

In [28]:
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path("/net/trapnell/vol1/home/mdcolon/proj/morphseq")
QC_DIR = PROJECT_ROOT / "results/mcolon/20260607_sci_cilia_gene14_imaging_qc"
HERE = PROJECT_ROOT / "results/mcolon/20260727_gene14_clean/0_shared"

MORPHSEQ_FILE = QC_DIR / "tables/query_all_rows_clean.csv"
PHENOTYPE_PREDICTIONS = QC_DIR / "predictions/sequenced_homozygous_phenotype_cross_bin.csv"
EXCEL_DIR = QC_DIR / "source_plate_metadata_excels"

SEQUENCE_METADATA = Path("/net/seahub_zfish/vol1/data/preprocessed/GENE14/GENE14_embryo_metadata.tsv")
SEQUENCE_QC = Path("/net/seahub_zfish/vol1/data/BBI_dmux_sci/GENE14/260625_GENE14_run2_novaseqx/GENE14/missing_embryos_list.csv")
MCCLINTOCK_EMBRYOS = HERE / "embryo_table.tsv"
OUTPUT_FILE = HERE / "morphseq_embryo_sequence_map.csv"
PREFERRED_LABELS_FILE = HERE / "preferred_morphseq_labels.csv"

# Manual decisions stay small, visible, and easy to audit.
MANUAL_EXCLUSIONS = {
    "20260319_cilia_crispant_18hpf_H02_e02": "Bubble segmented as a second embryo",
}
MANUAL_REVIEW_NOTES = {
    "20260414_sci_b9d2_48hpf_plate01_B11_e01": "One time-lapse track split across e01 and e02",
    "20260414_sci_b9d2_48hpf_plate01_B11_e02": "One time-lapse track split across e01 and e02",
}
MANUAL_PREFERRED_LABELS = {}

## 1. One row per MorphSeq embryo

The source table has one row per image frame. We keep only embryo-level columns and remove exact frame repeats.

In [29]:
embryo_columns = [
    "embryo_id",
    "experiment_id",
    "well",
    "physical_embryo_id",
    "gene",
    "collection_time_hpf",
    "data_source",
]

morphseq_rows = pd.read_csv(MORPHSEQ_FILE, usecols=embryo_columns, low_memory=False)

# Every frame belonging to one embryo must report the same imaging well.
wells_per_embryo = morphseq_rows.groupby("embryo_id")["well"].nunique(dropna=False)
assert wells_per_embryo.eq(1).all()

# These identity fields are constant across image frames.
# collection_time_hpf is added later from the authoritative Excel sheet.
morphseq_embryos = morphseq_rows.groupby("embryo_id", as_index=False).agg(
    experiment_id=("experiment_id", "first"),
    well=("well", "first"),
    physical_embryo_id=("physical_embryo_id", "first"),
    gene=("gene", "first"),
    data_source=("data_source", "first"),
)

# The well is written inside every MorphSeq embryo ID: ..._A01_e01
# Store it as morphseq_imaging_well for the Excel cross-reference. The well is also
# stored in the "well" column, so the cute little checks make sure they match. 🐟
morphseq_embryos["morphseq_imaging_well"] = morphseq_embryos["embryo_id"].str.extract(r"_([A-H]\d{2})_e\d+$", expand=False)

# One row per embryo, every ID parsed, and both versions of the well agree.
assert morphseq_embryos["embryo_id"].is_unique
assert morphseq_embryos["morphseq_imaging_well"].notna().all()
assert morphseq_embryos["morphseq_imaging_well"].eq(morphseq_embryos["well"].str.upper()).all()
morphseq_embryos = morphseq_embryos.drop(columns="well")

print(f"{len(morphseq_embryos):,} MorphSeq embryos from {morphseq_embryos['experiment_id'].nunique()} experiments")
display(morphseq_embryos.head())

1,643 MorphSeq embryos from 24 experiments


,embryo_id,experiment_id,physical_embryo_id,gene,data_source,morphseq_imaging_well
0,20260319_cilia_crispant_18hpf_A01_e01,20260319_cilia_crispant_18hpf,crispant_18hpf_plate01_A01,crispant,snapshot,A01
1,20260319_cilia_crispant_18hpf_A02_e01,20260319_cilia_crispant_18hpf,crispant_18hpf_plate01_A02,crispant,snapshot,A02
2,20260319_cilia_crispant_18hpf_A03_e01,20260319_cilia_crispant_18hpf,crispant_18hpf_plate01_A03,crispant,snapshot,A03
3,20260319_cilia_crispant_18hpf_A04_e01,20260319_cilia_crispant_18hpf,crispant_18hpf_plate01_A04,crispant,snapshot,A04
4,20260319_cilia_crispant_18hpf_A05_e01,20260319_cilia_crispant_18hpf,crispant_18hpf_plate01_A05,crispant,snapshot,A05


## 2. Add phenotype predictions

Every MorphSeq embryo stays in the table. Prediction fields are simply blank when that embryo was not classified.

In [30]:
phenotype_predictions = pd.read_csv(PHENOTYPE_PREDICTIONS, low_memory=False)[
    [
        "embryo_id",
        "predicted_label",
        "top_probability",
        "argmax_margin",
        "prob_CE",
        "prob_HTA",
        "prob_High_to_Low",
        "prob_Low_to_High",
    ]
].rename(
    columns={
        "predicted_label": "predicted_phenotype",
        "top_probability": "phenotype_top_probability",
        "argmax_margin": "phenotype_argmax_margin",
    }
)

assert phenotype_predictions["embryo_id"].is_unique

morphseq_embryos = morphseq_embryos.merge(phenotype_predictions, on="embryo_id", how="left", validate="one_to_one")

print(f"phenotype predictions: {morphseq_embryos['predicted_phenotype'].notna().sum():,}")

phenotype predictions: 169


## 3. Match each experiment to its plate Excel file

This deliberately boring dictionary is the seam between imaging and plate metadata. It is easy to read, edit, and audit.

In [31]:
experiment_to_excel = {
    "20260319_cilia_crispant_18hpf": "20260319_cilia_crispant_18hpf_well_metadata.xlsx",
    "20260319_cilia_crispant_24hpf": "20260319_cilia_crispant_24hpf_well_metadata.xlsx",
    "20260319_cilia_crispant_30hpf": "20260319_cilia_crispant_30hpf_well_metadata.xlsx",
    "20260320_cilia_crispant_48hpf": "20260320_cilia_crispant_48hpf_well_metadata.xlsx",
    "20260324_cep290_18hpf_24hpf_plate02": "20260324_cep290_18hpf_24hpf_plate02_well_metadata.xlsx",
    "20260324_cep290_18hpf_plate01": "20260324_cep290_18hpf_plate01_well_metadata.xlsx",
    "20260324_cep290_24hpf_plate01": "20260324_cep290_24hpf_plate01_well_metadata.xlsx",
    "20260324_cep290_24hpf_plate02": "20260324_cep290_24hpf_plate02_well_metadata.xlsx",
    "20260324_cep290_30hpf_plate01": "20260324_cep290_30hpf_plate01_well_metadata.xlsx",
    "20260324_cep290_30hpf_plate02": "20260324_cep290_30hpf_plate02_well_metadata.xlsx",
    "20260331_b9d2_18hpf_plate01": "20260331_b9d2_18hpf_plate01_well_metadata.xlsx",
    "20260331_b9d2_18hpf_plate02": "20260331_b9d2_18hpf_plate02_well_metadata.xlsx",
    "20260414_b9d2_14hpf_plate01": "20260414_b9d2_14hpf_plate01_well_metadata.xlsx",
    "20260414_b9d2_14hpf_plate02": "20260414_b9d2_14hpf_plate02_well_metadata.xlsx",
    "20260414_b9d2_30hpf_plate01": "20260414_b9d2_30hpf_plate01_well_metadata.xlsx",
    "20260414_b9d2_30hpf_plate02": "20260414_b9d2_30hpf_plate02_well_metadata.xlsx",
    "20260414_sci_b9d2_48hpf_plate01": "20260414_sci_b9d2_48hpf_plate01_well_metadata.xlsx",
    "20260415_b9d2_30to48hpf_plate01_t02": "20260415_b9d2_30to48hpf_plate01_t02_well_metadata.xlsx",
    "20260415_b9d2_30to48hpf_plate02_t02": "20260415_b9d2_30to48hpf_plate02_t02_well_metadata.xlsx",
    "20260415_cep290_18hpf_plate03": "20260415_cep290_18hpf_plate03_well_metadata.xlsx",
    "20260415_cep290_30to48hpf_plate02_t01": "20260415_cep290_30to48hpf_plate02_t01_well_metadata.xlsx",
    "20260415_sci_cep290_48hpf_plate01": "20260415_sci_cep290_48hpf_plate01_well_metadata.xlsx",
    "20260416_cep290_30to48hpf_plate01_t02": "20260416_cep290_30to48hpf_plate01_t02_well_metadata.xlsx",
    "20260416_cep290_30to48hpf_plate02_t02": "20260416_cep290_30to48hpf_plate02_t02_well_metadata.xlsx",
}

missing_experiments = sorted(set(morphseq_embryos["experiment_id"]) - set(experiment_to_excel)) #checks to make sure all embryos and expeiemnts have an associated excel file. If not, it will throw an error and print the missing experiments.
assert not missing_experiments, f"Experiments without an Excel file: {missing_experiments}"

missing_files = [name for name in experiment_to_excel.values() if not (EXCEL_DIR / name).exists()]
assert not missing_files, f"Excel files not found: {missing_files}"

print(f"{len(experiment_to_excel)} experiments mapped to Excel files")

24 experiments mapped to Excel files


## 4. Put every Excel sheet into long format

Each sheet becomes one column. Even an empty sheet is retained as a blank column. No biological decisions happen here—it is just reshaping an 8 × 12 plate.

In [32]:
plate_rows = list("ABCDEFGH")
plate_columns = [str(number) for number in range(1, 13)]
plate_wells = [f"{row}{int(column):02d}" for row in plate_rows for column in plate_columns]


def plate_sheet_values(grid):
    """Return the 96 plate cells in A01, A02, ... H12 order."""
    if grid.empty:
        return [pd.NA] * 96

    grid = grid.copy()
    grid.index = grid.index.astype(str).str.strip().str.upper()
    grid.columns = [str(column).strip() for column in grid.columns]
    # A Numbers-generated "Export Summary" is not a plate; it becomes a blank column.
    grid = grid.loc[~grid.index.duplicated()]
    grid = grid.reindex(index=plate_rows, columns=plate_columns)
    return grid.to_numpy().reshape(-1)


excel_tables = []

# Use the experiment-to-Excel dictionary to open the correct workbook.
# Each workbook becomes one 96-well table containing all of its sheets.
for experiment_id, excel_name in experiment_to_excel.items():
    excel_path = EXCEL_DIR / excel_name
    sheets = pd.read_excel(excel_path, sheet_name=None, index_col=0)

    plate = pd.DataFrame({"morphseq_imaging_well": plate_wells})
    for sheet_name, grid in sheets.items():
        plate[sheet_name] = plate_sheet_values(grid)
        # print(sheet_name, grid.shape, plate[sheet_name].notna().sum())

    plate["experiment_id"] = experiment_id
    plate["excel_file"] = excel_name
    excel_tables.append(plate)

excel_long = pd.concat(excel_tables, ignore_index=True)
assert not excel_long.duplicated(["experiment_id", "morphseq_imaging_well"]).any()

print(f"{len(excel_long):,} Excel well rows")
display(excel_long.head())

2,304 Excel well rows


,morphseq_imaging_well,notes,pair,genotype,sequenced,strain,chem_perturbation,start_age_hpf,qc,temperature,image_to_hash_map,hash_plate_num,rt_block,collection_time_hpf,experiment_id,excel_file,Export Summary,seqeneced,Sheet10,start_stage_hpf
0,A01,NaN,NaN,sspo_crispant,1.0,ab,NaN,18.0,NaN,28.5,A01,2.0,Bl1,18,20260319_cilia_crispant_18hpf,20260319_cilia_crispant_18hpf_well_metadata.xlsx,NaN,NaN,NaN,NaN
1,A02,NaN,NaN,sspo_crispant,1.0,ab,NaN,18.0,NaN,28.5,A02,2.0,Bl1,18,20260319_cilia_crispant_18hpf,20260319_cilia_crispant_18hpf_well_metadata.xlsx,NaN,NaN,NaN,NaN
2,A03,NaN,NaN,foxj1a_crispant,1.0,ab,NaN,18.0,NaN,28.5,A03,2.0,Bl1,18,20260319_cilia_crispant_18hpf,20260319_cilia_crispant_18hpf_well_metadata.xlsx,NaN,NaN,NaN,NaN
3,A04,NaN,NaN,ift88_crispant,1.0,ab,NaN,18.0,NaN,28.5,A04,2.0,Bl1,18,20260319_cilia_crispant_18hpf,20260319_cilia_crispant_18hpf_well_metadata.xlsx,NaN,NaN,NaN,NaN
4,A05,NaN,NaN,ift88_crispant,1.0,ab,NaN,18.0,NaN,28.5,A05,2.0,Bl1,18,20260319_cilia_crispant_18hpf,20260319_cilia_crispant_18hpf_well_metadata.xlsx,NaN,NaN,NaN,NaN


In [33]:
excel_long["rt_block"].unique()

array(['Bl1', nan, 'Bl2', 'Bl3', 'Bl4', 'Bl7', 'Bl6', 'Bl8', 'Bl9', 'Bl5'],
      dtype=object)

## 5. Join by experiment and imaging well

Now we use `(experiment_id, morphseq_imaging_well)` to attach the matching Excel metadata to each MorphSeq embryo.

A blank `image_to_hash_map` means the plate uses the same well on both sides. In that case, `A01 → A01`.

In [34]:
n_morphseq_embryos = len(morphseq_embryos)
morphseq_embryos = morphseq_embryos.merge(
    excel_long,
    on=["experiment_id", "morphseq_imaging_well"],
    how="left",
    validate="many_to_one",
)
assert len(morphseq_embryos) == n_morphseq_embryos

# Read the mapped hash well from Excel.
hash_well_from_excel = (
    morphseq_embryos["image_to_hash_map"].astype("string").str.strip().replace("", pd.NA) 
)
# A blank Excel cell means identity mapping: imaging A01 -> hash A01. ✨
morphseq_embryos["hash_well"] = hash_well_from_excel.fillna(
    morphseq_embryos["morphseq_imaging_well"]
)
# Sequencing IDs write A1, not the imaging-style A01.
morphseq_embryos["hash_well"] = morphseq_embryos["hash_well"].str.replace(
    r"^([A-H])0+(\d+)$", r"\1\2", regex=True
)


def format_hash_plate(value):
    """Turn Excel values such as 2, 2.0, or P02 into P02."""
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA
    text = str(value).strip().upper().removeprefix("P")
    return f"P{int(float(text)):02d}"


morphseq_embryos["hash_plate"] = morphseq_embryos["hash_plate_num"].map(format_hash_plate)

required_coordinates = ["hash_plate", "hash_well", "rt_block"]
has_coordinates = (
    morphseq_embryos[required_coordinates]
    .notna()
    .all(axis="columns") #generate a column if it has all the info to desingate a sequence_embryo_id label
)
morphseq_embryos["pred_seq_embryo_ID"] = pd.NA
morphseq_embryos.loc[has_coordinates, "pred_seq_embryo_ID"] = (
    "GENE14_"
    + morphseq_embryos.loc[has_coordinates, "hash_plate"].astype(str)
    + "_"
    + morphseq_embryos.loc[has_coordinates, "hash_well"].astype(str)
    + "_"
    + morphseq_embryos.loc[has_coordinates, "rt_block"].astype(str)
)

print(f"predicted sequencing IDs constructed: {morphseq_embryos['pred_seq_embryo_ID'].notna().sum():,}")

predicted sequencing IDs constructed: 1,643


## 6. Choose one phenotype label per sequencing embryo

The full map keeps every imaging record. This step makes a separate downstream table with one explicit label choice per sequencing embryo.

In [ ]:
# Record manual exclusions and review notes without deleting audit rows.
morphseq_embryos["manual_exclusion_reason"] = morphseq_embryos["embryo_id"].map(MANUAL_EXCLUSIONS)
morphseq_embryos["manual_review_note"] = morphseq_embryos["embryo_id"].map(MANUAL_REVIEW_NOTES)

# Only rows with a real phenotype prediction can supply a downstream label.
label_candidates = morphseq_embryos[
    morphseq_embryos["predicted_phenotype"].notna()
    & morphseq_embryos["pred_seq_embryo_ID"].notna()
    & morphseq_embryos["manual_exclusion_reason"].isna()
].copy()

# Name each acquisition so the preference is obvious.
label_candidates["label_source"] = "snapshot"
label_candidates.loc[
    label_candidates["experiment_id"].str.contains("_t01", na=False),
    "label_source",
] = "snapshot_t01"
label_candidates.loc[
    label_candidates["experiment_id"].str.contains("_t02", na=False),
    "label_source",
] = "snapshot_t02"
label_candidates.loc[
    label_candidates["data_source"].eq("timeseries"),
    "label_source",
] = "timeseries"

label_priority = {
    "timeseries": 1,
    "snapshot_t02": 2,
    "snapshot": 3,
    "snapshot_t01": 4,
}
label_candidates["label_priority"] = label_candidates["label_source"].map(label_priority)

# Choose one row per sequencing embryo. Equal-priority ties require a manual choice.
chosen_label_rows = []
for seq_embryo_id, candidates in label_candidates.groupby("pred_seq_embryo_ID"):
    manual_choice = MANUAL_PREFERRED_LABELS.get(seq_embryo_id)

    if manual_choice is not None:
        chosen = candidates[candidates["embryo_id"].eq(manual_choice)]
        if len(chosen) != 1:
            raise ValueError(f"Invalid manual label choice for {seq_embryo_id}: {manual_choice}")
        chosen = chosen.iloc[0].copy()
        chosen["selection_reason"] = "manual choice"
    else:
        best_priority = candidates["label_priority"].min()
        best_candidates = candidates[candidates["label_priority"].eq(best_priority)]
        if len(best_candidates) != 1:
            details = best_candidates[
                ["embryo_id", "label_source", "predicted_phenotype"]
            ].to_string(index=False)
            raise ValueError(
                f"Equal-priority phenotype labels for {seq_embryo_id}; "
                f"add a choice to MANUAL_PREFERRED_LABELS:\n{details}"
            )
        chosen = best_candidates.iloc[0].copy()
        chosen["selection_reason"] = f"preferred {chosen['label_source']}"

    chosen_label_rows.append(chosen)

preferred_morphseq_labels = pd.DataFrame(chosen_label_rows).rename(
    columns={"embryo_id": "chosen_morphseq_embryo_id"}
)
preferred_morphseq_labels = preferred_morphseq_labels[
    [
        "pred_seq_embryo_ID",
        "chosen_morphseq_embryo_id",
        "predicted_phenotype",
        "phenotype_top_probability",
        "label_source",
        "selection_reason",
    ]
].sort_values("pred_seq_embryo_ID").reset_index(drop=True)

assert preferred_morphseq_labels["pred_seq_embryo_ID"].is_unique



preferred_morphseq_labels.to_csv(PREFERRED_LABELS_FILE, index=False)
print(f"Wrote {PREFERRED_LABELS_FILE.name}: {len(preferred_morphseq_labels):,} labels")
display(preferred_morphseq_labels.head())

Wrote preferred_morphseq_labels.csv: 167 labels


,pred_seq_embryo_ID,chosen_morphseq_embryo_id,predicted_phenotype,phenotype_top_probability,label_source,selection_reason
0,GENE14_P02_A10_Bl5,20260415_sci_cep290_48hpf_plate01_A10_e01,High_to_Low,0.995226,timeseries,preferred timeseries
1,GENE14_P02_A10_Bl6,20260414_b9d2_14hpf_plate01_A10_e01,CE,0.662257,snapshot,preferred snapshot
2,GENE14_P02_A11_Bl3,20260324_cep290_24hpf_plate01_A11_e01,High_to_Low,0.726115,snapshot,preferred snapshot
3,GENE14_P02_A12_Bl4,20260324_cep290_30hpf_plate01_A12_e01,Low_to_High,0.999920,snapshot,preferred snapshot
4,GENE14_P02_A12_Bl7,20260331_b9d2_18hpf_plate01_A12_e01,HTA,0.998352,snapshot,preferred snapshot


### Mark the MorphSeq row to use downstream

We keep every MorphSeq row in the full map. When one sequencing embryo has multiple MorphSeq labels, `is_preferred_morphseq_embryo` marks the one label chosen above for downstream phenotype analysis.

In [ ]:
# True only for the MorphSeq rows chosen for downstream phenotype analysis.
chosen_morphseq_ids = preferred_morphseq_labels["chosen_morphseq_embryo_id"]
morphseq_embryos["is_preferred_morphseq_embryo"] = morphseq_embryos["embryo_id"].isin(
    chosen_morphseq_ids
)

## 7. Ask the sequencing tables what happened

These are annotations, not mapping inputs. This keeps “was sequenced,” “passed sequencing QC,” and “made it into McClintock” as three separate questions.

In [36]:
sequence_metadata = pd.read_table(
    SEQUENCE_METADATA,
    usecols=["embryo_ID"],
)
sequence_qc = pd.read_csv(
    SEQUENCE_QC,
    usecols=["embryo_ID", "pass"],
).rename(
    columns={
        "embryo_ID": "pred_seq_embryo_ID",
        "pass": "passed_sequence_qc",
    }
)
mcclintock = pd.read_csv(MCCLINTOCK_EMBRYOS, sep="\t", usecols=["embryo_ID"], low_memory=False)

assert sequence_qc["pred_seq_embryo_ID"].is_unique

morphseq_embryos["in_sequence_metadata"] = morphseq_embryos["pred_seq_embryo_ID"].isin(sequence_metadata["embryo_ID"])
morphseq_embryos["in_mcclintock"] = morphseq_embryos["pred_seq_embryo_ID"].isin(mcclintock["embryo_ID"])
morphseq_embryos = morphseq_embryos.merge(
    sequence_qc,
    on="pred_seq_embryo_ID",
    how="left",
    validate="many_to_one",
)

In [37]:
sequence_metadata.head()

,embryo_ID
0,GENE14_P02_A1_Bl1
1,GENE14_P02_B1_Bl1
2,GENE14_P02_C1_Bl1
3,GENE14_P02_D1_Bl1
4,GENE14_P02_E1_Bl1


## 8. Save the finished embryo table

No rows are filtered. Missing values remain visible so downstream analysis can make its own explicit choices.

In [ ]:
first_columns = [
    "experiment_id",
    "embryo_id",
    "morphseq_imaging_well",
    "physical_embryo_id",
    "gene",
    "collection_time_hpf",
    "data_source",
    "predicted_phenotype",
    "excel_file",
]
last_columns = [
    "hash_well",
    "hash_plate",
    "pred_seq_embryo_ID",
    "in_sequence_metadata",
    "passed_sequence_qc",
    "in_mcclintock",
]
middle_columns = [
    column
    for column in morphseq_embryos.columns
    if column not in first_columns + last_columns
]

morphseq_embryos = morphseq_embryos[first_columns + middle_columns + last_columns]
morphseq_embryos = morphseq_embryos.sort_values(["experiment_id", "morphseq_imaging_well", "embryo_id"]).reset_index(drop=True)

assert len(morphseq_embryos) == 1643
assert morphseq_embryos["embryo_id"].is_unique

morphseq_embryos.to_csv(OUTPUT_FILE, index=False)
print(f"Wrote {OUTPUT_FILE}")
print(f"{len(morphseq_embryos):,} embryos × {morphseq_embryos.shape[1]} columns")

Wrote /net/trapnell/vol1/home/mdcolon/proj/morphseq/results/mcolon/20260727_gene14_clean/0_shared/morphseq_embryo_sequence_map.csv
1,643 embryos × 40 columns


## Tiny audit

Just enough checking to make missing values and disagreements easy to see.

In [ ]:
print("Excel sequenced values:")
display(morphseq_embryos["sequenced"].value_counts(dropna=False).rename("n_embryos").to_frame())

print("Sequencing outcomes:")
display(
    morphseq_embryos[["in_sequence_metadata", "passed_sequence_qc", "in_mcclintock"]]
    .value_counts(dropna=False)
    .rename("n_embryos")
    .to_frame()
)

missing_coordinates = morphseq_embryos[
    morphseq_embryos["hash_plate_num"].isna() | morphseq_embryos["rt_block"].isna()
][["experiment_id", "embryo_id", "morphseq_imaging_well", "sequenced", "hash_plate_num", "rt_block"]]
print(f"Embryos missing hash plate or RT block: {len(missing_coordinates):,}")
display(missing_coordinates.head(20))

# Keep snapshot and time-lapse mappings separate so repeated imaging is visible.
embryo_id_map_columns = [
    "experiment_id",
    "embryo_id",
    "physical_embryo_id",
    "pred_seq_embryo_ID",
]

snapshot_embryo_id_map = (
    morphseq_embryos.loc[
        morphseq_embryos["data_source"].eq("snapshot"),
        embryo_id_map_columns,
    ]
    .sort_values("embryo_id")
    .reset_index(drop=True)
)

timeseries_embryo_id_map = (
    morphseq_embryos.loc[
        morphseq_embryos["data_source"].eq("timeseries"),
        embryo_id_map_columns,
    ]
    .sort_values("embryo_id")
    .reset_index(drop=True)
)

# Each MorphSeq embryo ID must occur exactly once in its map.
assert snapshot_embryo_id_map["embryo_id"].is_unique
assert timeseries_embryo_id_map["embryo_id"].is_unique

print(f"Snapshot embryo-ID map: {len(snapshot_embryo_id_map):,} unique IDs")
display(snapshot_embryo_id_map.head())
print(f"Time-lapse embryo-ID map: {len(timeseries_embryo_id_map):,} unique IDs")
display(timeseries_embryo_id_map.head())

# Show cases where multiple MorphSeq IDs point to the same sequencing embryo.
reused_snapshot_ids = snapshot_embryo_id_map[
    snapshot_embryo_id_map["pred_seq_embryo_ID"].duplicated(keep=False)
]
reused_timeseries_ids = timeseries_embryo_id_map[
    timeseries_embryo_id_map["pred_seq_embryo_ID"].duplicated(keep=False)
]

print(f"Snapshot rows sharing a sequencing ID: {len(reused_snapshot_ids):,}")
display(reused_snapshot_ids.head(20))
print(f"Time-lapse rows sharing a sequencing ID: {len(reused_timeseries_ids):,}")
display(reused_timeseries_ids.head(20))

Excel sequenced values:


,n_embryos
sequenced,
0.0,826
1.0,372
NaN,231
2.0,214


Sequencing outcomes:


n_embryos
in_sequence_metadata passed_sequence_qc in_mcclintock           
False                NaN                False               1048
True                 True               True                 566
                     False              False                 29

Embryos missing hash plate or RT block: 0


...

,experiment_id,embryo_id,physical_embryo_id,pred_seq_embryo_ID
0,20260414_sci_b9d2_48hpf_plate01,20260414_sci_b9d2_48hpf_plate01_A01_e01,b9d2_48hpf_plate01_A01,GENE14_P02_A1_Bl9
1,20260414_sci_b9d2_48hpf_plate01,20260414_sci_b9d2_48hpf_plate01_A02_e01,b9d2_48hpf_plate01_A02,GENE14_P02_A2_Bl9
2,20260414_sci_b9d2_48hpf_plate01,20260414_sci_b9d2_48hpf_plate01_A03_e01,b9d2_48hpf_plate01_A03,GENE14_P02_A3_Bl9
3,20260414_sci_b9d2_48hpf_plate01,20260414_sci_b9d2_48hpf_plate01_A04_e01,b9d2_48hpf_plate01_A04,GENE14_P02_A4_Bl9
4,20260414_sci_b9d2_48hpf_plate01,20260414_sci_b9d2_48hpf_plate01_A05_e01,b9d2_48hpf_plate01_A05,GENE14_P02_A5_Bl9


Snapshot rows sharing a sequencing ID: 110


,experiment_id,embryo_id,physical_embryo_id,pred_seq_embryo_ID
49,20260319_cilia_crispant_18hpf,20260319_cilia_crispant_18hpf_H02_e01,crispant_18hpf_plate01_H02,GENE14_P02_H2_Bl1
50,20260319_cilia_crispant_18hpf,20260319_cilia_crispant_18hpf_H02_e02,crispant_18hpf_plate01_H02,GENE14_P02_H2_Bl1
215,20260324_cep290_18hpf_24hpf_plate02,20260324_cep290_18hpf_24hpf_plate02_A06_e01,cep290_24hpf_plate02_A06,GENE14_P18_A6_Bl3
216,20260324_cep290_18hpf_24hpf_plate02,20260324_cep290_18hpf_24hpf_plate02_A07_e01,cep290_24hpf_plate02_A07,GENE14_P18_A7_Bl3
227,20260324_cep290_18hpf_24hpf_plate02,20260324_cep290_18hpf_24hpf_plate02_B06_e01,cep290_24hpf_plate02_B06,GENE14_P18_B6_Bl3
228,20260324_cep290_18hpf_24hpf_plate02,20260324_cep290_18hpf_24hpf_plate02_B07_e01,cep290_24hpf_plate02_B07,GENE14_P18_B7_Bl3
238,20260324_cep290_18hpf_24hpf_plate02,20260324_cep290_18hpf_24hpf_plate02_C06_e01,cep290_24hpf_plate02_C06,GENE14_P18_C6_Bl3
239,20260324_cep290_18hpf_24hpf_plate02,20260324_cep290_18hpf_24hpf_plate02_C07_e01,cep290_24hpf_plate02_C07,GENE14_P18_C7_Bl3
250,20260324_cep290_18hpf_24hpf_plate02,20260324_cep290_18hpf_24hpf_plate02_D06_e01,cep290_24hpf_plate02_D06,GENE14_P18_D6_Bl3
251,20260324_cep290_18hpf_24hpf_plate02,20260324_cep290_18hpf_24hpf_plate02_D07_e01,cep290_24hpf_plate02_D07,GENE14_P18_D7_Bl3


Time-lapse rows sharing a sequencing ID: 2


,experiment_id,embryo_id,physical_embryo_id,pred_seq_embryo_ID
22,20260414_sci_b9d2_48hpf_plate01,20260414_sci_b9d2_48hpf_plate01_B11_e01,b9d2_48hpf_plate01_B11,GENE14_P02_B11_Bl9
23,20260414_sci_b9d2_48hpf_plate01,20260414_sci_b9d2_48hpf_plate01_B11_e02,b9d2_48hpf_plate01_B11,GENE14_P02_B11_Bl9
